# Aprendizado de Máquina — Aula prática E1

## Análise de Agrupamento: $k$-médias

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

As onze aulas anteriores tinham todas a mesma forma: havia um $Y$, e o objetivo era
prevê-lo. Tudo o que construímos — risco, validação cruzada, viés e variância —
dependia de existir uma resposta certa contra a qual comparar.

Aqui não há $Y$.

> **Sem resposta certa, não há erro a minimizar, não há validação cruzada, e não
> há "o" agrupamento correto. Há critérios — e escolher entre eles é parte do
> problema, não uma etapa técnica.**

Esta aula é sobre o mais usado desses critérios e o algoritmo que o otimiza. É um
algoritmo de meia dúzia de linhas, que vamos escrever do zero antes de chamar o do
`scikit-learn` — e cujos três defeitos (ótimo local, $K$ arbitrário, geometria
esférica) são justamente o que precisa ser sabido para usá-lo bem.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- implementar o algoritmo de Lloyd e verificar que a WCSS nunca sobe;
- reconhecer que o resultado depende da inicialização e **medir** com que
  frequência ela estraga o agrupamento;
- usar cotovelo e silhueta para escolher $K$, e reconhecer os limites dos dois;
- exibir um caso em que o $k$-médias falha por motivo geométrico;
- ler um dendrograma e relacionar o corte dele com o $K$ do $k$-médias;
- aplicar o $k$-médias à compressão de imagem.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são os dois algoritmos de agrupamento do `scikit-learn`, a
silhueta, e o `linkage`/`dendrogram` do `scipy`, que é quem desenha o dendrograma.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.datasets import load_sample_image, make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. O critério, e o algoritmo que o otimiza

O $k$-médias procura a partição em $K$ grupos que minimiza a **soma de quadrados
dentro dos grupos** (WCSS):

$$\mathrm{WCSS} = \sum_{k=1}^{K} \sum_{i \in G_k} \|x_i - \bar{x}_k\|^2 .$$

Encontrar o mínimo global é NP-difícil. O algoritmo de **Lloyd** faz o óbvio e
alterna dois passos, cada um dos quais só pode diminuir a WCSS:

1. **atribuir** cada ponto ao centro mais próximo;
2. **recalcular** cada centro como a média dos pontos que lhe couberam.

In [ ]:
def lloyd(X, K, centros, n_iter=12):
    """Devolve o historico de (rotulos, centros, WCSS) a cada iteracao."""
    historico = []
    for _ in range(n_iter):
        d = ((X[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
        rotulos = d.argmin(axis=1)                       # passo 1: atribuir
        wcss = d[np.arange(len(X)), rotulos].sum()
        historico.append((rotulos.copy(), centros.copy(), wcss))
        novos = np.array([X[rotulos == k].mean(axis=0) if np.any(rotulos == k)
                          else centros[k] for k in range(K)])   # passo 2: recalcular
        if np.allclose(novos, centros):
            break
        centros = novos
    return historico


X, y_verdadeiro = make_blobs(n_samples=500, centers=4, cluster_std=1.1,
                             random_state=7)
rng = np.random.default_rng(3)
centros0 = X[rng.choice(len(X), 4, replace=False)]     # inicializacao ingenua
hist = lloyd(X, 4, centros0)

print(f"iteracoes ate convergir: {len(hist)}")
for i, (_, _, w) in enumerate(hist):
    print(f"   iteracao {i}: WCSS = {w:9.2f}")

In [ ]:
fig, axes = subplots(1, 4, figsize=(11, 2.9))
passos = [min(j, len(hist) - 1) for j in (0, 1, 2, len(hist) - 1)]
for ax, i in zip(axes, passos):
    rot, cen, w = hist[i]
    ax.scatter(X[:, 0], X[:, 1], c=rot, s=8, cmap="viridis", alpha=0.7)
    ax.scatter(cen[:, 0], cen[:, 1], marker="X", s=170, c="crimson", edgecolor="k")
    ax.set_title(f"iteracao {i}   WCSS = {w:.0f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

A WCSS cai a cada passo e para. Isso não é sorte: os dois passos **só podem
diminuí-la** — o passo 1 porque cada ponto vai para o centro mais próximo, o passo 2
porque a média é o ponto que minimiza a soma de quadrados de um conjunto. Como há um
número finito de partições, o algoritmo converge sempre, e em poucas iterações.

O que ele **não** garante é chegar ao mínimo global. Ele para no primeiro ponto em
que nenhum dos dois passos muda nada — um ótimo local.

In [ ]:
km = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=0).fit(X)
print(f"WCSS da nossa implementacao : {hist[-1][2]:.2f}")
print(f"WCSS do KMeans (inertia_)   : {km.inertia_:.2f}")
print(f"os dois agrupamentos coincidem? "
      f"{adjusted_rand_score(hist[-1][0], km.labels_) > 0.999}")

---
## 3. O ótimo local, medido

"Depende da inicialização" é fácil de dizer e fácil de medir. Vamos rodar o
algoritmo 200 vezes, com um único início sorteado ao acaso em cada uma, e olhar a
distribuição das WCSS finais.

In [ ]:
rng_i = np.random.default_rng(0)
finais = []
for _ in range(200):
    c0 = X[rng_i.choice(len(X), 4, replace=False)]
    finais.append(lloyd(X, 4, c0, n_iter=50)[-1][2])
finais = np.array(finais)

melhor = finais.min()
ruins = finais > melhor * 1.01
print(f"melhor WCSS encontrada : {melhor:.2f}")
print(f"pior                   : {finais.max():.2f}  (+{100*(finais.max()/melhor-1):.0f}%)")
print(f"quantas das 200 ficaram presas num otimo local pior: {ruins.sum()} ({ruins.mean():.1%})")

fig, ax = subplots(figsize=(5.2, 2.8))
ax.hist(finais, bins=40, color="steelblue")
ax.axvline(melhor, color="crimson", lw=1.5, label="melhor encontrada")
ax.set_xlabel("WCSS final"); ax.set_ylabel("frequencia"); ax.legend(fontsize=8)

Uma fração significativa das rodadas termina em um agrupamento pior — às vezes
muito pior. É por isso que ninguém roda o $k$-médias uma vez só.

As duas defesas do `scikit-learn` estão nos parâmetros padrão, e vale saber o que
cada uma faz. O `n_init` repete o algoritmo várias vezes e fica com a melhor WCSS.
O `init="k-means++"` sorteia os centros iniciais **espalhados**: o primeiro ao
acaso, e cada seguinte com probabilidade proporcional ao quadrado da distância ao
centro mais próximo já escolhido.

In [ ]:
linhas = []
for init in ["random", "k-means++"]:
    for n_init in [1, 10]:
        w = [KMeans(n_clusters=4, init=init, n_init=n_init, random_state=s).fit(X).inertia_
             for s in range(60)]
        linhas.append({"init": init, "n_init": n_init, "WCSS media": np.mean(w),
                       "pior WCSS": np.max(w),
                       "% de rodadas ruins": np.mean(np.array(w) > melhor * 1.01)})
pd.DataFrame(linhas).set_index(["init", "n_init"]).round(4)

> **A lição.** As duas defesas funcionam, e funcionam por caminhos diferentes: o
> `k-means++` melhora **cada** tentativa, escolhendo pontos de partida espalhados; o
> `n_init` compra bilhetes a mais. Combinadas, o problema praticamente desaparece.
>
> Os padrões atuais do `scikit-learn` já são `init="k-means++"` e `n_init="auto"`.
> Mas se você implementar o $k$-médias por conta própria — ou usar uma biblioteca
> mais antiga — a inicialização é a primeira coisa a conferir.

---
## 4. Quantos grupos? Cotovelo e silhueta

A WCSS **sempre** cai quando $K$ cresce: com $K = n$ ela é zero. Ela não pode,
portanto, escolher $K$ — só mostrar onde a queda deixa de compensar. É o método do
**cotovelo**.

A **silhueta** é uma medida diferente, e melhor: para cada ponto, compara a
distância média aos pontos do próprio grupo ($a_i$) com a distância média aos pontos
do grupo vizinho mais próximo ($b_i$):

$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)} \in [-1, 1].$$

Um $s_i$ perto de 1 quer dizer que o ponto está bem no seu grupo; perto de $-1$,
que ele estaria melhor no vizinho.

In [ ]:
Ks = range(2, 11)
wcss, silh = [], []
for K in Ks:
    m = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X)
    wcss.append(m.inertia_)
    silh.append(silhouette_score(X, m.labels_))

fig, (ax1, ax2) = subplots(1, 2, figsize=(7.8, 3.0))
ax1.plot(list(Ks), wcss, "o-", ms=4)
ax1.axvline(4, ls=":", color="green")
ax1.set_xlabel("K"); ax1.set_ylabel("WCSS"); ax1.set_title("cotovelo", fontsize=9)
ax2.plot(list(Ks), silh, "s-", ms=4, color="crimson")
ax2.axvline(list(Ks)[int(np.argmax(silh))], ls=":", color="green")
ax2.set_xlabel("K"); ax2.set_ylabel("silhueta media"); ax2.set_title("silhueta", fontsize=9)

print(f"K que maximiza a silhueta: {list(Ks)[int(np.argmax(silh))]}")
print(f"K verdadeiro (nos geramos os dados): {len(np.unique(y_verdadeiro))}")

Neste exemplo os dois concordam com a verdade — mas repare na assimetria de
evidência. O cotovelo é uma inspeção visual, e "onde a curva dobra" é notoriamente
ambíguo em dados reais. A silhueta devolve um número, e um número tem máximo.

O diagnóstico mais útil, porém, não é a silhueta **média** e sim a distribuição
dela por grupo:

In [ ]:
m4 = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
s = silhouette_samples(X, m4.labels_)

fig, ax = subplots(figsize=(5.4, 3.4))
inicio = 0
for k in range(4):
    vals = np.sort(s[m4.labels_ == k])
    ax.barh(np.arange(inicio, inicio + len(vals)), vals, height=1.0)
    ax.text(-0.05, inicio + len(vals) / 2, f"grupo {k}", fontsize=8, ha="right")
    inicio += len(vals) + 12
ax.axvline(s.mean(), color="crimson", ls="--", lw=1.2, label=f"media = {s.mean():.3f}")
ax.set_xlabel("silhueta"); ax.set_yticks([]); ax.legend(fontsize=8)

print("silhueta media por grupo:")
for k in range(4):
    print(f"   grupo {k}: {s[m4.labels_ == k].mean():.3f}  ({int((m4.labels_ == k).sum())} pontos)")
print(f"pontos com silhueta negativa: {int((s < 0).sum())}")

Aqui os quatro grupos são igualmente coesos e nenhum ponto tem silhueta negativa —
o que é uma informação, e você só a tem porque olhou. O valor do gráfico está nos
casos em que ele **não** sai assim: uma média alta pode esconder um grupo largo e
mal definido no meio de três excelentes, e os pontos de silhueta negativa são
candidatos a estarem no grupo errado. O exercício abaixo constrói um caso desses.

Até aqui os grupos tinham dispersões parecidas. Vamos quebrar essa simetria: quatro
centros, três apertados e um muito espalhado.

In [ ]:
Xw, yw = make_blobs(n_samples=600, centers=4, cluster_std=[0.4, 0.4, 3.0, 0.4],
                    random_state=7)

Ks = range(2, 9)
inercias_w, silhuetas_w = [], []
for K in Ks:
    mw = KMeans(n_clusters=K, n_init=10, random_state=0).fit(Xw)
    inercias_w.append(mw.inertia_)
    silhuetas_w.append(silhouette_score(Xw, mw.labels_))

print(f"{'K':>3} {'inercia':>12} {'silhueta':>10}")
for K, i_w, s_w in zip(Ks, inercias_w, silhuetas_w):
    print(f"{K:3d} {i_w:12.1f} {s_w:10.4f}")
print(f"\nsilhueta escolhe K = {list(Ks)[int(np.argmax(silhuetas_w))]}")

m4w = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Xw)
print(f"\ncom K=4: ARI contra a verdade = {adjusted_rand_score(yw, m4w.labels_):.4f}")
print("tamanho de cada grupo VERDADEIRO :", np.bincount(yw))
print("tamanho de cada grupo ENCONTRADO :", np.bincount(m4w.labels_))
sw = silhouette_samples(Xw, m4w.labels_)
print("\nsilhueta media por grupo encontrado:")
for k in range(4):
    print(f"   grupo {k}: {sw[m4w.labels_ == k].mean():+.4f}  ({(m4w.labels_ == k).sum()} pontos)")

fig, (ax1, ax2) = subplots(1, 2, figsize=(9.6, 3.6))
ax1.scatter(Xw[:, 0], Xw[:, 1], c=yw, cmap="tab10", s=9)
ax1.set_title("os 4 grupos verdadeiros", fontsize=9)
ax2.scatter(Xw[:, 0], Xw[:, 1], c=m4w.labels_, cmap="tab10", s=9)
ax2.scatter(*m4w.cluster_centers_.T, c="black", marker="X", s=90)
ax2.set_title("o que o k-medias encontra com K=4", fontsize=9)
for ax in (ax1, ax2):
    ax.set_xticks([]); ax.set_yticks([])

**A silhueta escolhe $K=4$**, com $0{,}8175$ — e o cotovelo também: a inércia cai de
$19\,497$ para $2\,442$ até $K=4$ e depois se arrasta ($1\,575$, $1\,144$, $922$).
Os dois critérios concordam, e concordam com a verdade: o ARI contra os rótulos
verdadeiros é $0{,}9693$.

Isso é menos óbvio do que parece. O $k$-médias minimiza a soma de quadrados dentro
dos grupos, e um grupo três vezes mais espalhado contribui com muito mais soma que
os outros três juntos — o incentivo natural seria quebrá-lo em dois e fundir dois
apertados. **Não é o que acontece**, porque os três apertados estão longe uns dos
outros: fundi-los custaria mais do que quebrar o largo economiza.

O preço aparece em outro lugar, e é o gráfico por grupo que o mostra. As silhuetas
médias são $+0{,}56$, $+0{,}86$, $+0{,}93$ e $+0{,}91$ — o grupo largo vale a metade
dos outros. E os tamanhos denunciam o resto: os quatro grupos verdadeiros têm 150
pontos cada, e os encontrados têm 143, 155, 152 e 150. O $k$-médias **encolhe** o
grupo largo, entregando sete dos seus pontos de borda aos vizinhos apertados —
porque a fronteira que ele traça fica sempre a meio caminho entre dois centros,
independentemente de um grupo ser mais espalhado que o outro.

É a limitação estrutural do método: fronteiras equidistantes, portanto grupos
esféricos e de tamanho parecido. Quando isso não vale, ele não erra o número de
grupos — erra as bordas.

---
## 5. O defeito que nenhum $K$ conserta

O critério da WCSS não é neutro: minimizar soma de quadrados a partir de centros
produz grupos **esféricos e de tamanho parecido**. Quando os grupos verdadeiros têm
outra forma, nenhuma escolha de $K$ salva.

In [ ]:
Xl, yl = make_moons(n_samples=500, noise=0.06, random_state=2)
Xa, ya = make_blobs(n_samples=[400, 100, 100], centers=[[0, 0], [5, 5], [5, 0]],
                    cluster_std=[2.2, 0.4, 0.4], random_state=1)

fig, axes = subplots(1, 4, figsize=(11, 2.9))
for ax, (Xd, yd, nome) in zip(axes[:2], [(Xl, yl, "duas luas"),
                                         (Xa, ya, "tamanhos diferentes")]):
    ax.scatter(Xd[:, 0], Xd[:, 1], c=yd, s=8, cmap="viridis")
    ax.set_title(f"{nome}: a verdade", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
for ax, (Xd, yd, nome, K) in zip(axes[2:], [(Xl, yl, "duas luas", 2),
                                            (Xa, ya, "tamanhos diferentes", 3)]):
    rot = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(Xd)
    ax.scatter(Xd[:, 0], Xd[:, 1], c=rot, s=8, cmap="viridis")
    ax.set_title(f"{nome}: k-medias (ARI={adjusted_rand_score(yd, rot):.2f})", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

Nas duas luas o $k$-médias corta ao meio, porque a fronteira dele entre dois centros
é sempre uma **reta** (a mediatriz). Nos grupos de tamanhos diferentes, ele reparte
o grupo grande e junta pedaços dos pequenos, porque a WCSS pune muito um grupo
espalhado.

O índice Rand ajustado (ARI) mede a concordância com a verdade, corrigida pelo
acaso: 1 é concordância perfeita, 0 é o esperado ao acaso. Aqui só usamos porque
inventamos os dados — em agrupamento de verdade não existe `y` para comparar, e é
exatamente por isso que este notebook insiste em olhar as figuras.

---
## 6. Agrupamento hierárquico e o dendrograma

O $k$-médias exige $K$ de antemão. O agrupamento **hierárquico aglomerativo** não:
começa com cada ponto no seu próprio grupo e vai fundindo os dois mais próximos até
sobrar um. O resultado é uma árvore — o **dendrograma** — e o $K$ vira uma escolha
de **onde cortar**, feita depois de ver a estrutura.

In [ ]:
sub = np.random.default_rng(0).choice(len(X), 60, replace=False)
Z = linkage(X[sub], method="ward")

fig, (ax1, ax2) = subplots(1, 2, figsize=(11, 3.4))
dendrogram(Z, ax=ax1, no_labels=True, color_threshold=Z[-3, 2])
ax1.axhline(Z[-3, 2], ls="--", color="crimson", label="corte em 4 grupos")
ax1.set_ylabel("distancia da fusao"); ax1.legend(fontsize=8)
ax1.set_title("dendrograma (ligacao de Ward, 60 pontos)", fontsize=9)

rot_h = AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(X)
ax2.scatter(X[:, 0], X[:, 1], c=rot_h, s=8, cmap="viridis")
ax2.set_title(f"hierarquico, 4 grupos "
              f"(ARI com o k-medias = {adjusted_rand_score(rot_h, m4.labels_):.3f})",
              fontsize=9)
ax2.set_xticks([]); ax2.set_yticks([])
print(f"altura das 5 ultimas fusoes: {np.round(Z[-5:, 2], 1)}")

A altura de cada fusão é a distância em que ela aconteceu, e um salto grande entre
alturas consecutivas sugere onde cortar. É a versão hierárquica do cotovelo — com a
vantagem de você ver a estrutura inteira antes de decidir.

A **ligação** é o parâmetro que define "distância entre dois grupos": *ward* (a que
minimiza o aumento da WCSS, e por isso se parece com o $k$-médias), *complete* (a
maior distância entre pares), *average*, *single* (a menor — que produz grupos
alongados e é a única das quatro que resolveria as duas luas).

In [ ]:
linhas = []
for lig in ["ward", "complete", "average", "single"]:
    rot = AgglomerativeClustering(n_clusters=2, linkage=lig).fit_predict(Xl)
    linhas.append({"ligacao": lig, "ARI nas duas luas": adjusted_rand_score(yl, rot)})
linhas.append({"ligacao": "k-medias",
               "ARI nas duas luas": adjusted_rand_score(
                   yl, KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xl))})
pd.DataFrame(linhas).set_index("ligacao").round(4)

A ligação *single* resolve as duas luas perfeitamente, e todas as outras falham —
porque ela só exige que exista **um caminho** de vizinhos próximos entre os pontos,
que é exatamente a estrutura de uma lua. Em compensação, ela é frágil: um único
ponto entre os dois grupos os funde.

A moral, de novo, é que não há método universal. O critério que você escolhe é uma
hipótese sobre a forma dos grupos.

---
## 7. Padronizar não é opcional

Pela última vez no curso: a WCSS soma coordenadas ao quadrado. Uma variável em
unidade grande domina o agrupamento inteiro.

In [ ]:
Xs = X.copy()
Xs[:, 1] *= 100                                  # a segunda coluna em outra unidade

rot_sem = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(Xs)
rot_com = KMeans(n_clusters=4, n_init=10,
                 random_state=0).fit_predict(StandardScaler().fit_transform(Xs))

print(f"ARI com a verdade, SEM padronizar: {adjusted_rand_score(y_verdadeiro, rot_sem):.4f}")
print(f"ARI com a verdade, COM padronizar: {adjusted_rand_score(y_verdadeiro, rot_com):.4f}")

E note a diferença em relação à classificação: aqui **não há validação cruzada**
para pegar o erro. Um agrupamento ruim por falta de padronização não gera nenhum
aviso — ele simplesmente produz grupos, e os grupos parecem grupos.

---
## 8. Uma aplicação: comprimir uma imagem

Uma imagem colorida é uma tabela: cada *pixel* é uma observação com três
covariáveis (R, G, B). Agrupar os pixels em $K$ grupos e substituir cada um pela cor
do seu centro é **quantização de cor** — uma compressão com perda, e uma das
aplicações mais visuais do $k$-médias.

In [ ]:
imagem = load_sample_image("china.jpg")
altura, largura, _ = imagem.shape
pixels = (imagem.reshape(-1, 3) / 255.0)
print(f"imagem: {altura} x {largura} pixels")
print(f"cores distintas na original: {len(np.unique(pixels, axis=0)):,}")

In [ ]:
amostra = pixels[np.random.default_rng(0).choice(len(pixels), 20_000, replace=False)]

fig, axes = subplots(1, 4, figsize=(11, 2.8))
axes[0].imshow(imagem); axes[0].set_title("original", fontsize=9); axes[0].axis("off")
for ax, K in zip(axes[1:], [2, 8, 32]):
    km_img = KMeans(n_clusters=K, n_init=4, random_state=0).fit(amostra)
    nova = km_img.cluster_centers_[km_img.predict(pixels)].reshape(imagem.shape)
    erro = np.mean((nova - pixels.reshape(imagem.shape)) ** 2)
    ax.imshow(nova)
    ax.set_title(f"K = {K} cores\nEQM = {erro:.5f}", fontsize=9); ax.axis("off")

In [ ]:
bits_original = altura * largura * 24
for K in [2, 8, 32, 256]:
    bits = altura * largura * np.ceil(np.log2(K)) + K * 24
    print(f"K = {K:3d} cores: {bits/8/1024:8.1f} KB   "
          f"({bits/bits_original:.1%} do original)")
print(f"original (24 bits/pixel): {bits_original/8/1024:.1f} KB")

Com 32 cores a imagem ainda é perfeitamente reconhecível e ocupa cerca de um quinto
do espaço. É o mesmo balanço de sempre, com outro nome: $K$ controla a
complexidade, e a EQM contra a imagem original é o "erro de treino" — que, como
sempre, só cai quando $K$ cresce.

A diferença é que aqui **não existe erro de teste**: não há imagem nova para
prever. O que decide o $K$ é uma pessoa olhando o resultado e um orçamento de
armazenamento. É o retrato do aprendizado não supervisionado.

A mesma compressão na outra imagem de exemplo, e a comparação que interessa: o $K$
que o olho aceita contra o $K$ que a silhueta escolheria nos pixels.

In [ ]:
flor = load_sample_image("flower.jpg")
alt_f, larg_f, _ = flor.shape
px_f = flor.reshape(-1, 3) / 255.0
rng_f = np.random.default_rng(0)
amostra_px = px_f[rng_f.choice(len(px_f), 3000, replace=False)]

print(f"imagem: {alt_f} x {larg_f}, {len(np.unique(px_f, axis=0)):,} cores distintas\n")
print(f"{'K':>4} {'EQM da reconstrucao':>21} {'KB':>9} {'% do original':>15}")
bits_orig_f = alt_f * larg_f * 24
for K in (2, 4, 8, 16, 32, 64):
    mk = KMeans(n_clusters=K, n_init=4, random_state=0).fit(amostra_px)
    recon = mk.cluster_centers_[mk.predict(px_f)]
    bits = alt_f * larg_f * np.ceil(np.log2(K)) + K * 24
    print(f"{K:4d} {np.mean((px_f - recon) ** 2):21.6f} {bits/8/1024:9.1f} "
          f"{bits/bits_orig_f:14.1%}")

print(f"\nsilhueta nos pixels (amostra de 3000):")
for K in (2, 3, 4, 5, 6, 8):
    mk = KMeans(n_clusters=K, n_init=4, random_state=0).fit(amostra_px)
    print(f"   K = {K}: {silhouette_score(amostra_px, mk.labels_):+.4f}")

fig, axes = subplots(1, 4, figsize=(11, 3.0))
for ax, K in zip(axes, (2, 8, 32, None)):
    if K is None:
        ax.imshow(flor); ax.set_title("original", fontsize=9)
    else:
        mk = KMeans(n_clusters=K, n_init=4, random_state=0).fit(amostra_px)
        ax.imshow(mk.cluster_centers_[mk.predict(px_f)].reshape(alt_f, larg_f, 3))
        ax.set_title(f"K = {K}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

| $K$ | EQM da reconstrução | KB | % do original |
| --- | --- | --- | --- |
| 2 | $0{,}011063$ | $33{,}4$ | 4,2% |
| 8 | $0{,}002246$ | $100{,}1$ | 12,5% |
| 16 | $0{,}001151$ | $133{,}5$ | 16,7% |
| 32 | $0{,}000676$ | $166{,}9$ | 20,8% |
| 64 | $0{,}000419$ | $200{,}3$ | 25,0% |

Olhando as quatro imagens, $K=2$ é claramente insuficiente e $K=32$ já é difícil de
distinguir do original a olho — 21% do tamanho, com $62\,941$ cores reduzidas a 32.
Entre 16 e 32 fica a fronteira do que cada um enxerga.

**E a silhueta escolheria $K=2$**, com $+0{,}7657$, caindo monotonamente daí em
diante ($+0{,}71$, $+0{,}50$, $+0{,}48$, $+0{,}44$). Um fator de oito a dezesseis de
diferença.

Os dois números não têm por que coincidir porque **as perguntas são outras**. A
silhueta pergunta *"os pontos formam grupos bem separados?"* — e a nuvem de cores
de uma foto não forma: é um contínuo, com no máximo duas regiões densas (o céu
claro e o resto). Ela responde corretamente que quase não há estrutura de grupos.

A compressão pergunta *"quantos representantes bastam para o erro de reconstrução
ficar imperceptível?"* — e isso não exige grupo nenhum. Bastam pontos bem espalhados
pela nuvem, cada um cobrindo a sua vizinhança. É quantização vetorial, não análise
de agrupamento; o algoritmo é o mesmo e o critério de sucesso não.

A moral vale para além de imagens: **antes de escolher $K$ por uma métrica, saiba
para que o $K$ serve.** Silhueta, cotovelo e gap statistic medem separação. Se o seu
uso é resumir, comprimir ou pré-processar, o critério certo é o erro da tarefa que
vem depois.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| algoritmo de Lloyd | §2 | dois passos, e a WCSS não pode subir em nenhum deles |
| ótimo local | §3 | com um início ao acaso, uma fração das rodadas termina pior |
| `k-means++` e `n_init` | §3 | defesas diferentes: uma melhora cada tentativa, a outra compra bilhetes |
| cotovelo × silhueta | §4 | a WCSS sempre cai; a silhueta tem máximo — e vale olhá-la por grupo |
| geometria | §5 | a fronteira entre dois centros é sempre reta: as duas luas são impossíveis |
| dendrograma | §6 | $K$ escolhido **depois** de ver a estrutura; a ligação é uma hipótese sobre a forma |
| ligação *single* | §6 | resolve as duas luas, que todas as outras erram |
| padronização | §7 | sem ela o agrupamento é decidido pela unidade de medida — e nada avisa |
| compressão | §8 | 32 cores, um quinto do espaço, e nenhum "erro de teste" para consultar |

**Leitura recomendada.** [AME] Capítulo 11 (análise de agrupamento), em especial a
discussão sobre por que não há critério objetivo para escolher $K$. [ISLP] §12.4:
§12.4.1 ($k$-médias, com a mesma figura de iterações da nossa Seção 2), §12.4.2
(hierárquico e as ligações) e §12.4.3, que é uma lista honesta das decisões
arbitrárias que um agrupamento exige — vale ler inteira antes de reportar grupos
para alguém.

**Para praticar.** `Lista teorica E1.pdf` (teórica, com gabarito) e
`Lista prática E1.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula E2 continua sem $Y$, mas com outro objetivo: em vez de agrupar
observações, **reduzir colunas** — encontrar as poucas direções em que os dados de
fato variam. É a ferramenta que a Aula 05 pediu quando falamos de redundância.